# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 4 — LightGBM Feature-Based Forecaster

**Duration** 50 minutes · **Work in pairs** · `git checkout lab4-solution`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 5 | Find the **two leaks** in the starter code you inherited. Write them down in `LEAKS.md`. **Do not fix them yet.** |
| 12 | Build the leakage-safe design matrix with `build_features` — ACF-chosen lags, shifted rolling windows, KSA calendar flags, Fourier terms. Verify with `assert_no_leakage`. |
| 12 | Direct multi-step targets for H = 24 with `DirectLGBMForecaster`. Split **by time**. Say out loud why direct, not recursive. |
| 8 | Forecast the test window day by day. MAE and MAPE against the seasonal-naive baseline; overlay one day. |
| 8 | Feature importance by gain and by family. Interpret the top five. Importance is associational, never causal. |
| 5 | The four-design leakage demonstration. Finish `LEAKS.md`. Commit. |

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar

df = join_calendar(load_demand(f"{DATA}/ksa_grid_demand.csv"),
                   load_calendar(f"{DATA}/ksa_calendar.csv"))
y = df["demand_mw"].astype(float)
print(f"{len(df):,} hourly rows   {df.index.min():%Y-%m-%d} -> {df.index.max():%Y-%m-%d}")

### Task 1 — the starter code you inherited

Run the next cell. It works, and the number it prints would get a model promoted.

Your job for the next five minutes is **not** to fix it. It is to name, precisely, the
two lines that make that number meaningless — and write them into `LEAKS.md`.

In [ ]:
# ---------------------------------------------------------------------------
# The starter code you inherited
#
# A colleague left this behind on their last day, with the note:
#   "day-ahead demand model, MAPE well under 2%, ready for the control room."
#
# It runs. It reports that number. Two lines in this cell make it a lie.
# Find both before you write any new code.
# ---------------------------------------------------------------------------
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# Target: demand at hour t. Day-ahead means every feature must be known 24 hours
# before t — so the lags all start at 24. That much the colleague got right.
work = df[["demand_mw", "temp_c"]].copy()
work["hour"] = work.index.hour
work["dow"]  = work.index.dayofweek
for L in (24, 25, 48, 168):
    work[f"lag{L}"] = work["demand_mw"].shift(L)
work["roll24"]     = work["demand_mw"].rolling(24, center=True, min_periods=1).mean()
work["roll24_max"] = work["demand_mw"].rolling(24, center=True, min_periods=1).max()
work = work.dropna()

FEATS = ["temp_c", "hour", "dow", "lag24", "lag25", "lag48", "lag168",
         "roll24", "roll24_max"]
Xa, Xb, ya, yb = train_test_split(work[FEATS], work["demand_mw"],
                                  test_size=0.2, random_state=0, shuffle=True)

starter = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.06, num_leaves=63,
                            verbose=-1, random_state=1).fit(Xa, ya)
starter_mape = float(np.mean(np.abs((yb - starter.predict(Xb)) / yb)) * 100)
print(f"inherited starter — day-ahead MAPE: {starter_mape:.2f}%")

**Leak 1 — the split.** `train_test_split(..., shuffle=True)` draws the test set
uniformly at random from three years of hourly data. Roughly 80% of the test rows sit
*earlier* in time than the last training row, and every test hour has its immediate
neighbours in the training set. The model is being asked to interpolate inside a period
it has already seen, which is not the question the control room is asking.

**Leak 2 — the feature.** `rolling(24, center=True)` puts the window *around* `t`, not
behind it: with an even window pandas centres it on hours `t-12 … t+11`. The column named
"the average of the past 24 hours" contains the **target hour itself** and eleven hours of
its future. `roll24_max` is worse — it hands over the peak. The lags are all 24 or more, so
the day-ahead framing is right; one mis-specified window destroys it anyway.

The next cell proves both rather than asserting them.

In [ ]:
# Proof 1 — the "test set" is not in the future
print("train window :", f"{Xa.index.min():%Y-%m-%d} -> {Xa.index.max():%Y-%m-%d}")
print("test  window :", f"{Xb.index.min():%Y-%m-%d} -> {Xb.index.max():%Y-%m-%d}")
n_before = int((Xb.index < Xa.index.max()).sum())
print(f"test rows earlier than the last training row   : {n_before:,} of {len(Xb):,} "
      f"({n_before/len(Xb):.0%})")
train_stamps = set(Xa.index)
both = sum((t - pd.Timedelta(1, "h")) in train_stamps and
           (t + pd.Timedelta(1, "h")) in train_stamps for t in Xb.index)
print(f"test rows with BOTH neighbouring hours in train: {both:,} of {len(Xb):,} "
      f"({both/len(Xb):.0%})")

# Proof 2 — the centred window looks forward, past the target hour
t = work.index[12000]
win = work.loc[t - pd.Timedelta(12, "h"): t + pd.Timedelta(11, "h"), "demand_mw"]
print()
print(f"origin t                                   : {t:%Y-%m-%d %H:%M}")
print(f'work["roll24"] at t                        : {work.loc[t, "roll24"]:,.1f}')
print(f"mean of demand over t-12h .. t+11h          : {win.mean():,.1f}")
print(f'work["roll24_max"] at t                    : {work.loc[t, "roll24_max"]:,.1f}')
print(f"max  of demand over t-12h .. t+11h          : {win.max():,.1f}")
print(f'the target itself, demand at t             : {work.loc[t, "demand_mw"]:,.1f}')
print()
print("A feature the model could only compute 11 hours after the hour it predicts.")

**`LEAKS.md`, section 1**

```
## Leaks in the inherited starter (labs/lab4)

1. Shuffled split — `train_test_split(..., shuffle=True)` on an hourly series.
   About 80% of the test rows precede the last training row, and every test hour
   is bracketed by training hours. The reported error measures interpolation
   inside seen history, not forecasting.
   Fix: split by time. Train through 2023-09-30, test after it.

2. Centred rolling window — `rolling(24, center=True)` on demand_mw.
   At hour t the window spans t-12 .. t+11, so "the past 24-hour average"
   contains the target hour ITSELF and eleven hours of its future. roll24_max
   leaks the peak outright.
   Fix: .shift(1) BEFORE .rolling(..., center=False), so the window ends at t-1.
```

### Task 2 — the leakage-safe design matrix

Four families, all computable at the forecast origin:

- **lags** — the significant ACF lags from Lab 2: 1, 2, 3 (short memory), 24, 25
  (yesterday's same hour and the hour after), 48, 168, 169, 336 (the weekly rhythm).
- **rolling** — `.shift(1)` **before** `.rolling(..., center=False)`, windows of 24 and 168 hours.
- **calendar** — hour, day of week, month, day of year, and the KSA flags. The weekend here is
  **Friday (4) and Saturday (5)** in pandas `dayofweek`, where Monday = 0. A model built with the
  Western Sat/Sun assumption learns the wrong two days.
- **Fourier** — smooth daily (period 24), weekly (168) and annual (8760) seasonality in a
  handful of sine/cosine columns instead of hundreds of dummies.

In [ ]:
from tayyar.features.build import (build_features, assert_no_leakage,
                                    DEFAULT_LAGS, DEFAULT_WINDOWS)

print("lags    :", DEFAULT_LAGS)
print("windows :", DEFAULT_WINDOWS)

feat = build_features(df[["demand_mw", "temp_c", "is_weekend",
                          "is_ramadan", "is_eid", "is_national_day"]])
Xcols = [c for c in feat.columns if c != "demand_mw"]
X, yy = feat[Xcols].astype(float), feat["demand_mw"]

print(f"\ndesign matrix : {len(feat):,} rows x {len(Xcols)} features")
print("families      :",
      {"lag":     sum("_lag" in c for c in Xcols),
       "rolling": sum("_r" in c and c.startswith("demand") for c in Xcols),
       "fourier": sum(c[:2] in ("d_", "w_", "y_") for c in Xcols),
       "weather": sum(c in ("temp_c", "cdd", "hdd", "temp_lag24") for c in Xcols)})

In [ ]:
# The two things that make it safe, checked rather than assumed.

# 1. the rolling window ends at t-1
manual = yy.shift(1).rolling(24, min_periods=6, center=False).mean()
print("rmean24 == shift(1).rolling(24).mean() :",
      bool(np.allclose(manual.dropna(), feat["demand_mw_rmean24"].dropna())))

# 2. the KSA weekend is Friday and Saturday
wk = feat.groupby("day_of_week")["is_weekend"].mean()
print("is_weekend by dayofweek (Mon=0)        :", wk.round(2).to_dict())

# 3. no feature is a disguised copy of the target, and no past feature moves
#    when the future is shuffled
assert_no_leakage(feat[["demand_mw"] + Xcols])

`assert_no_leakage` does two things. The cheap check is a correlation screen — no feature
may sit at |r| ≈ 1 with the target. The check that actually matters is the second one: it
shuffles the second half of the target, rebuilds the features, and demands that every
feature in the **first** half is unchanged. A feature that moves when the future moves is
a feature that reads the future. The centred window from Task 1 fails this instantly.

The long lags cost rows: `demand_mw_lag336` is undefined for the first 336 hours, so those
rows drop out of training. That is expected and correct — it is the price of a two-week
memory, not a bug.

### Task 3 — direct multi-step, split by time

**Recursive** is one model that predicts one step ahead and is then fed its own prediction
back in as a lag, 24 times. It is compact and it guarantees a coherent path — but the errors
compound, and after a few steps the "lags" it is consuming are predictions, whose
distribution is not the distribution of the real lags it was trained on.

**Direct** is one model per horizon `h`, each trained on the target shifted by `-h`. Nothing
compounds, because nothing is fed back. The costs are `H` models to fit and store, and no
guarantee that the 24 horizons form a coherent day — hour 13 is not constrained to sit
sensibly next to hour 14.

At a 24-hour day-ahead horizon the compounding is exactly where recursive chains fall apart,
so **direct is the robust default here**. `DirectLGBMForecaster` is direct.

In [ ]:
from tayyar.models.lgbm_forecaster import DirectLGBMForecaster

CUT_TRAIN = "2023-09-30 23:00"          # everything after this hour is unseen

Xtr = X[:CUT_TRAIN].dropna()
ytr = yy[Xtr.index]
Xte = X[CUT_TRAIN:]

print(f"train : {Xtr.index.min():%Y-%m-%d} -> {Xtr.index.max():%Y-%m-%d}   ({len(Xtr):,} rows)")
print(f"test  : {Xte.index.min():%Y-%m-%d} -> {Xte.index.max():%Y-%m-%d}   ({len(Xte):,} rows)")
print(f"no training hour is later than the test window :",
      bool(Xtr.index.max() <= Xte.index.min()))
print(f"overlap : {len(Xtr.index.intersection(Xte.index))} hour — the cut hour itself. "
      f"pandas label slicing is inclusive at BOTH ends; state that, never assume it.")

lgbm = DirectLGBMForecaster(horizon=24).fit(Xtr, ytr)
print(f"\nfitted {len(lgbm.models_)} models — one per horizon h = 1..24")
print("horizon h is trained on y.shift(-h): the target is h hours AHEAD of the features.")

23,707 training rows out of 26,280 hours: the 336-hour lag and the 2023-10-01 cut
account for the difference. The fit is 24 separate LightGBM models and takes a minute or two.

The one-hour overlap is worth pausing on. `X[:"2023-09-30 23:00"]` and `X["2023-09-30 23:00":]`
both contain the cut hour, because pandas label slicing is closed at both ends. It is harmless
here — the first forecast origin in Task 4 is 1 November — but an off-by-one at a split
boundary is precisely the class of defect this course is about, and the only defence is to
print it rather than assume it.

### Task 4 — forecast the test window, one day at a time

One forecast origin per day at 23:00, 60 days of November and December 2023. At each origin
the model sees one row of features and emits 24 hours. The baseline is seasonal-naive:
tomorrow is a copy of the last 24 hours. `MASE < 1` means you beat that free baseline.

In [ ]:
from tayyar.eval.metrics import all_metrics

test_idx = X[CUT_TRAIN:].index
starts = list(range(len(test_idx) - 25, 24 * 30, -24))[:60][::-1]

rows_model, rows_base, days = [], [], []
for s in starts:
    origin = test_idx[s]
    pos = X.index.get_loc(origin)
    Xo = X.iloc[[pos]]
    actual = yy.iloc[pos + 1: pos + 25]
    if len(actual) < 24 or Xo.isna().any().any():
        continue
    fc = np.array([lgbm.models_[h].predict(Xo)[0] for h in range(1, 25)])
    last24 = yy.iloc[pos - 23: pos + 1].to_numpy()
    bl = np.array([last24[h % 24] for h in range(24)])
    hist = yy.iloc[max(0, pos - 24 * 365): pos]          # in-sample scale for MASE
    rows_model.append(all_metrics(actual, fc, hist, 24))
    rows_base.append(all_metrics(actual, bl, hist, 24))
    days.append({"origin": origin, "actual": actual, "fc": fc, "bl": bl})

res = pd.DataFrame({"LightGBM (direct)": pd.DataFrame(rows_model).mean(),
                    "Seasonal-naive":    pd.DataFrame(rows_base).mean()}).T
print(f"{len(days)} daily origins, "
      f"{days[0]['origin']:%Y-%m-%d} -> {days[-1]['origin']:%Y-%m-%d}\n")
print(res[["MAE", "RMSE", "MAPE_%", "WAPE_%", "MASE", "bias"]].round(3))

In [ ]:
k = 20
d = days[k]
tt = pd.date_range(d["origin"] + pd.Timedelta(1, "h"), periods=24, freq="h")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(tt, d["actual"].to_numpy(), color="#1F3864", lw=2.0, label="actual")
ax.plot(tt, d["fc"], color="#C00000", lw=1.8, label="LightGBM (direct)")
ax.plot(tt, d["bl"], color="#7F7F7F", lw=1.2, ls="--", label="seasonal-naive")
ax.set_title(f"Day-ahead forecast from origin {d['origin']:%Y-%m-%d %H:%M}")
ax.set_ylabel("MW"); ax.legend(frameon=False, ncols=3, fontsize=8)
plt.tight_layout()

**Expected**

```
                     MAE      RMSE   MAPE_%  WAPE_%   MASE    bias
LightGBM (direct)  719.075   904.303  2.328   2.385   0.445  -14.060
Seasonal-naive    1244.025  1464.803  4.117   4.189   0.770   74.795
```

A MASE of 0.445 means the model makes about 45% of the error of the free baseline. Note the
baseline's own MASE is 0.77, not 1.0 — the MASE denominator is the seasonal-naive error
computed *in-sample* over the previous year, and December is a calmer month than the summer
that dominates that year. Baselines are not constants; always report the one you actually ran.

Also note the bias: -14 MW on a ~30,000 MW system, so the model is essentially unbiased
across the window. The seasonal-naive baseline is not — it lags a rising series.

### Task 5 — what the model leaned on

In [ ]:
from tayyar.models.explain import importance_by_family

imp = lgbm.feature_importance(25)          # mean GAIN across the 24 horizon models
print(imp.head(10).to_string(index=False))
print()
print(importance_by_family(imp).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.6))
top = imp.head(14).iloc[::-1]
ax.barh(top.feature, top.gain, color="#1F3864")
ax.set_title("LightGBM feature importance (mean gain over 24 horizon models)")
ax.set_xlabel("mean gain")
plt.tight_layout()

**Read the top five aloud.**

- `demand_mw_lag3` — the level three hours back. The single strongest anchor: whatever the
  grid was doing recently is where tomorrow starts from.
- `d_cos1` — the first daily cosine. A smooth, continuous encoding of time-of-day that lets a
  tree cut the day into shape rather than into 24 unrelated buckets.
- `demand_mw_rmean168` — the trailing weekly mean. The slow level: heat wave, industrial ramp,
  seasonal drift.
- `hour` — the raw hour, cutting the sharp edges the Fourier terms smooth over (the evening ramp).
- `demand_mw_rmean24` — yesterday's trailing mean. The short-run level.

Rolled up: rolling 33.1%, lags 29.0%, Fourier 25.8%, calendar 7.3%, weather 4.9%. Roughly
two-thirds of the gain is the series' own recent history; the calendar and Fourier terms shape
it; temperature adds under 5% *given* those (it is largely already encoded in the recent level).

**Say this in the room, every time.** Gain is *associational*. It measures how much this model,
with this feature set, on this data, reduced its loss by splitting on a column. It is not a
causal effect, it is not a sensitivity, and it does not survive a change in the feature set —
drop `demand_mw_rmean24` and `demand_mw_lag3` will simply absorb its gain. Two correlated
features split one column's importance between them and both look weak. Never hand a gain
table to an engineer as "what drives demand".

### The leakage demonstration — one model, four evaluation designs

Same LightGBM, same data, same day-ahead target. Only the *evaluation design* changes, plus
one feature. `tools/leak_demo.py` produces these; the recorded run is in `leak_results.json`.

In [ ]:
import json

leak = json.load(open("../leak_results.json"))
designs = [
    ("A", "shuffle split + leaky features", leak["A_shuffle_plus_leaky_features"]),
    ("B", "shuffle split, clean features",  leak["B_shuffle_clean_features"]),
    ("C", "time-ordered split, clean",      leak["C_time_split_clean"]),
    ("D", "rolling-origin backtest, clean", leak["D_rolling_origin_clean"]),
]
for k, name, v in designs:
    print(f"{k}  {name:<34s} {v:5.2f}% MAPE")
print(f"\nD spread across origins: {leak['D_spread'][0]:.2f}% .. {leak['D_spread'][1]:.2f}%")
print(f"the leaked design reports a model "
      f"{leak['D_rolling_origin_clean'] / leak['A_shuffle_plus_leaky_features']:.1f}x "
      f"more accurate than it actually is.")

fig, ax = plt.subplots(figsize=(8.6, 4))
labs = [f"{k}  {n}".replace(" + ", "\n+ ").replace(", ", ",\n") for k, n, _ in designs]
vals = [v for _, _, v in designs]
bars = ax.bar(labs, vals, color=["#A11B2A", "#C2601B", "#D9A441", "#1E7A4D"], width=.62)
for r, v in zip(bars, vals):
    ax.text(r.get_x() + r.get_width() / 2, v + .06, f"{v:.2f}%",
            ha="center", fontweight="bold")
ax.set_ylabel("day-ahead MAPE (%)"); ax.set_ylim(0, max(vals) * 1.25)
ax.set_title("One model, one dataset, one target — four evaluation designs")
plt.tight_layout()

**A -> B** is the cost of the leaky features alone: 1.61% -> 2.00%, still on a shuffled split.

**B -> C** is the cost of the split alone: 2.00% -> 3.54%. The split is the larger of the two
frauds, and it is the one that leaves no trace in the feature code.

**C -> D** is honesty about variance: a single time-ordered split is one anecdote (3.54%);
sixty rolling origins say 3.66% with a spread from 2.31% to 4.89%. The spread is the number
you quote to a regulator, because it is the range the control room will actually live in.

**A vs D** is the headline: the design that would have been reported is **2.3x** better than
the truth. Nothing in the model was wrong. Everything in the evaluation was.

(These four numbers use a plain single-horizon LightGBM rather than the direct 24-model
forecaster of Task 3, which is why D at 3.66% is a little worse than the 2.33% you measured
above. The comparison that matters is between the four designs, not against Task 4.)

### Mini-quiz — answers

1. **Why is `train_test_split(shuffle=True)` fatal?** Because test points can precede
   training points. The model is scored on hours it effectively saw the neighbourhood of.
   It leaks the future into the past, and no amount of feature hygiene repairs it.
2. **What does `.shift(1)` before rolling prevent?** The window including the target hour
   itself — self-leak. `y.rolling(24).mean()` at time `t` contains `y[t]`; `y.shift(1).rolling(24).mean()`
   ends at `t-1`, which is all you have at the forecast origin.
3. **Which compounds errors?** Recursive. Each step is fed its own previous prediction as a lag,
   so an error at step 1 is an input error at step 2.
4. **Why can a boosted tree never forecast a record high?** Trees partition the training range and
   predict a constant in each leaf. They interpolate; they cannot extrapolate beyond the range they
   were trained on. A new all-time peak is outside every leaf value.
5. **What tells you which lags to include?** The ACF — the significant lags. Here: 1, 2, 3
   (short memory), 24, 25 (yesterday), 48, 168, 169, 336 (the weekly rhythm).

### Task 6 — `LEAKS.md` and commit

`LEAKS.md` has two sections by now:

1. **The two leaks in the inherited starter**, each with the line that causes it and the
   mechanism in one sentence.
2. **The four-design table** above, with the sentence a reviewer needs:
   *the number that would have been reported to the control room was 2.3x better than the
   truth, and nothing in the code looked wrong.*

```
git add -A && git commit -m "feat(features): leakage-safe design matrix + direct LightGBM day-ahead forecaster"
```

### Instructor notes

**Protect the full 50 minutes.** With Lab 6 this is one of the two most overrun-prone slots in
the course. If you are behind, cut Task 5's chart, never Task 1.

- **Run Task 1 as a race.** First pair to name both leaks *precisely* — the line and the
  mechanism, not "something's wrong with the split" — wins. It takes the room four minutes
  and it is the highest-retention four minutes of Day 2. Do not hint before minute three.
- **Branch `sim-centred-roll`** ships the centred window as the only defect, with an
  otherwise-correct time split. Use it with a fast group: the MAPE is merely *good* rather than
  absurd, which is how this defect actually arrives in production.
- **Branch `sim-heatwave`** holds out a week containing a record high. The raw-target model
  under-forecasts the peak by a wide margin and the room can see the ceiling in the plot —
  trees interpolate, they cannot extrapolate past the training range. Two legitimate
  mitigations: model a differenced or relative target (percentage of the trailing weekly mean,
  say) so the extrapolation happens in the level rather than the tree; or accept the ceiling
  and keep a classical/level model in the candidate set for exactly these weeks. Do not let
  anyone "fix" it by adding more trees.
- **"What does this feature see?"** Put four columns on the board — `demand_mw_lag1`,
  `demand_mw_rmean24`, `is_eid`, `temp_c` — and have each pair rule each one legal or illegal at
  a 24-hour origin. `temp_c` is the deliberately ambiguous one: it is legal *only* if you
  substitute a day-ahead weather **forecast** in production, and the docstring in
  `features/build.py` says so. A model validated on realised temperature and deployed on
  forecast temperature will quietly degrade. This argument is worth five minutes.

**Troubleshooting**

| symptom | cause |
|---|---|
| MAPE below 0.7% | a leak survived. Look for a centred window, a `shift(-h)` in the features, or a scaler fitted on everything. |
| model under-forecasts a heat spike | not a bug. Trees cannot extrapolate beyond the training range. |
| most rows disappeared | `demand_mw_lag336` leaves 336 leading NaNs. Expected — drop them, never back-fill. |
| weekend flag on the wrong days | pandas `dayofweek` is Monday=0. The KSA weekend is 4 (Friday) and 5 (Saturday). |
| `assert_no_leakage` reports a different feature count | a boolean column slipped in; cast the flags to int. |

**Fast finishers.** Refit with `objective="l2"` instead of the default `"l1"` and compare the
bias on the peak hours; or drop the whole rolling family and watch the lag family absorb its
gain — the cleanest possible demonstration that importance is not a causal quantity.